In [51]:
!pip -q install pandas pyarrow numpy tqdm sentence-transformers faiss-cpu rank-bm25 groq google-generativeai

In [52]:
import os
import re
import json
import pickle
import numpy as np
import pandas as pd

from google.colab import drive, userdata

import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from groq import Groq
import google.generativeai as genai

In [53]:
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [54]:
PROJECT_DIR = "/content/drive/MyDrive/finance-rag-analyst"

INDEX_DIR = f"{PROJECT_DIR}/data/indexes"
EVAL_DIR = f"{PROJECT_DIR}/data/evaluation"
ANSWER_DIR = f"{PROJECT_DIR}/data/answers"

os.makedirs(ANSWER_DIR, exist_ok=True)

CHUNK_METADATA_PATH = f"{INDEX_DIR}/chunk_metadata.parquet"
EMBEDDINGS_PATH = f"{INDEX_DIR}/bge_small_embeddings.npy"
FAISS_INDEX_PATH = f"{INDEX_DIR}/faiss_bge_small.index"
BM25_INDEX_PATH = f"{INDEX_DIR}/bm25_index.pkl"
BEST_CONFIG_PATH = f"{EVAL_DIR}/best_retrieval_config.json"

SAMPLE_ANSWERS_PATH = f"{ANSWER_DIR}/sample_rag_answers.json"

print("Paths ready.")

Paths ready.


Load retrieval artifacts

In [55]:
for path in [
    CHUNK_METADATA_PATH,
    EMBEDDINGS_PATH,
    FAISS_INDEX_PATH,
    BM25_INDEX_PATH,
    BEST_CONFIG_PATH,
]:
    assert os.path.exists(path), f"Missing required artifact: {path}"

chunks_df = pd.read_parquet(CHUNK_METADATA_PATH)
embeddings = np.load(EMBEDDINGS_PATH)
faiss_index = faiss.read_index(FAISS_INDEX_PATH)

with open(BM25_INDEX_PATH, "rb") as f:
    bm25_payload = pickle.load(f)

bm25_index = bm25_payload["bm25_index"]
tokenized_corpus = bm25_payload["tokenized_corpus"]

with open(BEST_CONFIG_PATH, "r") as f:
    best_config_payload = json.load(f)

BEST_RETRIEVAL_CONFIG = best_config_payload["best_config"]

chunks_df = chunks_df.reset_index(drop=True)
chunks_df["row_id"] = chunks_df.index
chunks_df["text"] = chunks_df["text"].astype(str)
chunks_df["ticker"] = chunks_df["ticker"].astype(str).str.upper().str.strip()
chunks_df["topic_labels"] = chunks_df["topic_labels"].fillna("general").astype(str)

assert embeddings.shape[0] == len(chunks_df), "Embedding/chunk count mismatch."
assert faiss_index.ntotal == len(chunks_df), "FAISS/chunk count mismatch."
assert len(tokenized_corpus) == len(chunks_df), "BM25/chunk count mismatch."

print("Chunks:", len(chunks_df))
print("Best retrieval config:")
print(json.dumps(BEST_RETRIEVAL_CONFIG, indent=2))

display(chunks_df.head())

Chunks: 2093
Best retrieval config:
{
  "mode": "hybrid_rrf",
  "dense_top_k": 30,
  "bm25_top_k": 20,
  "rrf_k": 30,
  "final_top_k": 10,
  "expand": true,
  "diversify": true
}


,row_id,chunk_id,document_id,ticker,company,form_type,filing_date,report_date,accession_number,source_url,chunk_index,chunk_char_start,chunk_char_end,char_count,word_count,estimated_tokens,topic_labels,text
0,0,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,10-K,2026-02-25,2026-01-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,0,0,3970,3970,648,842,regulation,nvda-20260125 Table of Contents UNITED STATES ...
1,1,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,10-K,2026-02-25,2026-01-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,1,3370,7228,3858,584,759,advertising|cybersecurity|mda|regulation|risk_...,required a recovery analysis of incentive-base...
2,2,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,10-K,2026-02-25,2026-01-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,2,6628,10508,3880,574,746,advertising|ai_strategy|data_centers|regulatio...,dance at upcoming investor and industry confer...
3,3,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,10-K,2026-02-25,2026-01-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,3,9908,13722,3814,550,715,advertising|ai_strategy|business|data_centers|...,"mation may be limited or incomplete, and our s..."
4,4,NVDA_10-K_2026-02-25_000104581026000021_chunk_...,NVDA_10-K_2026-02-25_000104581026000021,NVDA,NVIDIA CORP,10-K,2026-02-25,2026-01-25,0001045810-26-000021,https://www.sec.gov/Archives/edgar/data/104581...,4,13122,17064,3942,598,777,acquisitions|advertising|ai_strategy|business|...,rkets with the same underlying technology by u...


Load embedding model

In [56]:
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Loaded:", EMBEDDING_MODEL_NAME)
print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded: BAAI/bge-small-en-v1.5
Embedding dimension: 384


/tmp/ipykernel_438/216677849.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", embedding_model.get_sentence_embedding_dimension())


Provider-flexible LLM setup

In [57]:
# Options: "groq", "gemini"
LLM_PROVIDER = "groq"

DEFAULT_GROQ_MODEL = "llama-3.1-8b-instant"
DEFAULT_GEMINI_MODEL = "gemini-1.5-flash"

try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)

assert groq_client is not None or GEMINI_API_KEY is not None, (
    "Add at least one key in Colab Secrets: GROQ_API_KEY or GEMINI_API_KEY."
)

print("LLM setup complete.")
print("Selected provider:", LLM_PROVIDER)
print("Groq available:", groq_client is not None)
print("Gemini available:", GEMINI_API_KEY is not None)

LLM setup complete.
Selected provider: groq
Groq available: True
Gemini available: True


Query expansion

In [58]:
QUERY_EXPANSIONS = {
    "ai": ["artificial intelligence", "generative ai", "machine learning", "accelerated computing"],
    "capex": ["capital expenditures", "capital investment", "infrastructure investment", "data centers"],
    "cloud": ["aws", "azure", "google cloud", "cloud services"],
    "chips": ["semiconductor", "gpu", "accelerator", "blackwell"],
    "china": ["greater china", "export controls", "trade restrictions"],
    "risk": ["risk factors", "could adversely affect", "uncertainty"],
    "advertising": ["ads", "ad revenue", "advertising revenue"],
}


def has_word(text, word):
    return re.search(rf"\b{re.escape(word)}\b", text.lower()) is not None


def expand_query(query, enabled=True):
    if not enabled:
        return query

    lowered = query.lower()
    expanded_terms = []

    for key, values in QUERY_EXPANSIONS.items():
        if has_word(lowered, key):
            expanded_terms.extend(values)

    if expanded_terms:
        return query + " " + " ".join(sorted(set(expanded_terms)))

    return query

Company intent detection

In [59]:
COMPANY_ALIASES = {
    "NVDA": ["nvidia", "nvda"],
    "MSFT": ["microsoft", "msft", "azure"],
    "AAPL": ["apple", "aapl", "iphone", "mac"],
    "AMZN": ["amazon", "amzn", "aws"],
    "GOOGL": ["google", "alphabet", "googl", "youtube", "google cloud"],
}


def detect_tickers_from_query(query):
    lowered = query.lower()
    detected = []

    for ticker, aliases in COMPANY_ALIASES.items():
        for alias in aliases:
            if re.search(rf"\b{re.escape(alias.lower())}\b", lowered):
                detected.append(ticker)
                break

    return sorted(set(detected))


def resolve_ticker_filter(question, user_tickers=None, auto_detect=True):
    if user_tickers is not None:
        return sorted(set([t.upper().strip() for t in user_tickers]))

    if auto_detect:
        detected = detect_tickers_from_query(question)
        if detected:
            return detected

    return None

Retrieval utilities

In [60]:
def tokenize_for_bm25(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9$%.\-]+", " ", text)
    return [tok for tok in text.split() if len(tok) > 1]


def build_filter_mask(tickers=None, forms=None):
    mask = np.ones(len(chunks_df), dtype=bool)

    if tickers is not None:
        tickers = [t.upper().strip() for t in tickers]
        mask &= chunks_df["ticker"].isin(tickers).values

    if forms is not None:
        forms = [f.strip() for f in forms]
        mask &= chunks_df["form_type"].isin(forms).values

    return mask


def make_result_row(row, rank, score, mode, source_modes=None):
    return {
        "rank": rank,
        "row_id": int(row["row_id"]),
        "chunk_id": row["chunk_id"],
        "ticker": row["ticker"],
        "company": row["company"],
        "form_type": row["form_type"],
        "filing_date": row["filing_date"],
        "topic_labels": row["topic_labels"],
        "retrieval_mode": mode,
        "retrieval_score": float(score),
        "source_modes": source_modes or mode,
        "source_url": row["source_url"],
        "text": row["text"],
        "text_preview": row["text"][:700],
    }

Dense, BM25, and hybrid retrieval

In [61]:
def dense_search(query, top_k=10, tickers=None, forms=None, expand=True, search_multiplier=5):
    expanded_query = expand_query(query, enabled=expand)

    query_emb = embedding_model.encode(
        [expanded_query],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")

    mask = build_filter_mask(tickers=tickers, forms=forms)

    candidate_k = len(chunks_df) if tickers is not None or forms is not None else min(
        len(chunks_df), max(top_k * search_multiplier, top_k)
    )

    scores, row_ids = faiss_index.search(query_emb, candidate_k)

    filtered = []
    for row_id, score in zip(row_ids[0], scores[0]):
        if row_id >= 0 and mask[row_id]:
            filtered.append((int(row_id), float(score)))
        if len(filtered) >= top_k:
            break

    rows = []
    for rank, (row_id, score) in enumerate(filtered, start=1):
        row = chunks_df.iloc[row_id]
        rows.append(make_result_row(row, rank, score, "dense"))

    return pd.DataFrame(rows)


def bm25_search(query, top_k=10, tickers=None, forms=None, expand=True):
    expanded_query = expand_query(query, enabled=expand)
    query_tokens = tokenize_for_bm25(expanded_query)

    scores = bm25_index.get_scores(query_tokens)
    mask = build_filter_mask(tickers=tickers, forms=forms)
    valid_indices = np.where(mask)[0]

    if len(valid_indices) == 0:
        return pd.DataFrame()

    valid_scores = scores[valid_indices]
    sorted_local = np.argsort(valid_scores)[::-1][:top_k]
    top_row_ids = valid_indices[sorted_local]

    rows = []
    for rank, row_id in enumerate(top_row_ids, start=1):
        row = chunks_df.iloc[row_id]
        rows.append(make_result_row(row, rank, float(scores[row_id]), "bm25"))

    return pd.DataFrame(rows)


def reciprocal_rank_fusion(result_dfs, rrf_k=60):
    fused_scores = {}
    source_modes = {}

    for df in result_dfs:
        if df is None or len(df) == 0:
            continue

        mode = df["retrieval_mode"].iloc[0]

        for _, row in df.iterrows():
            row_id = int(row["row_id"])
            rank = int(row["rank"])
            fused_scores[row_id] = fused_scores.get(row_id, 0.0) + 1.0 / (rrf_k + rank)
            source_modes.setdefault(row_id, []).append(mode)

    sorted_items = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    return sorted_items, source_modes


def diversify_results(sorted_items, final_top_k=10, max_chunks_per_document=3, max_chunks_per_ticker=5):
    selected = []
    doc_counts = {}
    ticker_counts = {}

    for row_id, score in sorted_items:
        row = chunks_df.iloc[row_id]
        doc_id = row["document_id"]
        ticker = row["ticker"]

        if doc_counts.get(doc_id, 0) >= max_chunks_per_document:
            continue
        if ticker_counts.get(ticker, 0) >= max_chunks_per_ticker:
            continue

        selected.append((row_id, score))
        doc_counts[doc_id] = doc_counts.get(doc_id, 0) + 1
        ticker_counts[ticker] = ticker_counts.get(ticker, 0) + 1

        if len(selected) >= final_top_k:
            break

    return selected


def hybrid_search(
    query,
    dense_top_k=20,
    bm25_top_k=20,
    final_top_k=10,
    rrf_k=60,
    tickers=None,
    forms=None,
    expand=True,
    diversify=True,
    max_chunks_per_document=3,
    max_chunks_per_ticker=5,
):
    dense_df = dense_search(query, dense_top_k, tickers, forms, expand)
    bm25_df = bm25_search(query, bm25_top_k, tickers, forms, expand)

    sorted_items, source_modes = reciprocal_rank_fusion([dense_df, bm25_df], rrf_k=rrf_k)

    selected = diversify_results(
        sorted_items,
        final_top_k=final_top_k,
        max_chunks_per_document=max_chunks_per_document,
        max_chunks_per_ticker=max_chunks_per_ticker
    ) if diversify else sorted_items[:final_top_k]

    rows = []
    for rank, (row_id, score) in enumerate(selected, start=1):
        row = chunks_df.iloc[row_id]
        modes = "|".join(sorted(set(source_modes.get(row_id, []))))
        rows.append(make_result_row(row, rank, score, "hybrid_rrf", source_modes=modes))

    return pd.DataFrame(rows)

Best-config retrieval wrapper

In [62]:
def retrieve_context(
    question,
    tickers=None,
    forms=None,
    final_top_k=None,
    auto_detect_tickers=True,
):
    resolved_tickers = resolve_ticker_filter(
        question,
        user_tickers=tickers,
        auto_detect=auto_detect_tickers
    )

    config = BEST_RETRIEVAL_CONFIG.copy()

    if final_top_k is not None:
        config["final_top_k"] = final_top_k

    mode = config.get("mode", "hybrid_rrf")

    if mode == "dense_only":
        results = dense_search(
            question,
            top_k=config.get("final_top_k", 10),
            tickers=resolved_tickers,
            forms=forms,
            expand=config.get("expand", True),
        )
    elif mode == "bm25_only":
        results = bm25_search(
            question,
            top_k=config.get("final_top_k", 10),
            tickers=resolved_tickers,
            forms=forms,
            expand=config.get("expand", True),
        )
    else:
        results = hybrid_search(
            question,
            dense_top_k=config.get("dense_top_k", 20),
            bm25_top_k=config.get("bm25_top_k", 20),
            final_top_k=config.get("final_top_k", 10),
            rrf_k=config.get("rrf_k", 60),
            tickers=resolved_tickers,
            forms=forms,
            expand=config.get("expand", True),
            diversify=config.get("diversify", True),
        )

    return results, resolved_tickers, config

Evidence adequacy guardrails

In [87]:
EXPLICIT_ENTITY_TERMS = [
    "india", "china", "japan", "korea", "taiwan", "vietnam",
    "europe", "european union", "uk", "united kingdom",
    "canada", "mexico", "brazil",
    "openai", "blackwell", "aws", "azure", "youtube",
]

QUERY_ASPECT_KEYWORDS = {
    "investment": [
        "investment", "investments", "invest", "invested", "capital investment",
        "capital expenditures", "capex", "infrastructure investment"
    ],
    "risk": [
        "risk", "risks", "uncertainty", "adverse", "regulatory", "restriction",
        "restrictions", "compliance"
    ],
    "revenue": [
        "revenue", "revenues", "sales", "net sales", "monetization"
    ],
    "ai": [
        "ai", "artificial intelligence", "generative ai", "machine learning",
        "accelerated computing"
    ],
    "competition": [
        "competition", "compete", "competitive", "competitor", "competitors"
    ],
    "cloud": [
        "cloud", "aws", "azure", "google cloud", "cloud services"
    ],
}


def contains_term(text, term):
    return re.search(rf"\b{re.escape(term.lower())}\b", str(text).lower()) is not None


def detect_explicit_entity_terms(question):
    return sorted({
        term for term in EXPLICIT_ENTITY_TERMS
        if contains_term(question, term)
    })


def detect_query_aspects(question):
    q = question.lower()
    detected = []

    for aspect, keywords in QUERY_ASPECT_KEYWORDS.items():
        if any(contains_term(q, kw) for kw in keywords):
            detected.append(aspect)

    return sorted(set(detected))


def chunk_matches_terms(text, terms):
    if not terms:
        return True
    return any(contains_term(text, term) for term in terms)


def chunk_matches_aspects(text, aspects):
    if not aspects:
        return True

    for aspect in aspects:
        keywords = QUERY_ASPECT_KEYWORDS.get(aspect, [])
        if any(contains_term(text, kw) for kw in keywords):
            return True

    return False


def enforce_evidence_adequacy(question, retrieved_df, resolved_tickers=None):
    """
    Guardrail:
    If the user asks about explicit entities/locations/products like India,
    keep only chunks that mention those entities.

    If the user asks about a specific aspect like investment, revenue, risk, etc.,
    require the remaining chunks to also mention that aspect.

    This prevents chunks about "India competition" from being used to answer
    "investment in India."
    """
    explicit_terms = detect_explicit_entity_terms(question)
    query_aspects = detect_query_aspects(question)

    if retrieved_df is None or len(retrieved_df) == 0:
        return retrieved_df, {
            "explicit_terms": explicit_terms,
            "query_aspects": query_aspects,
            "filtered": False,
            "reason": "no_retrieved_chunks",
            "ticker_support": {}
        }

    # If no special entity/aspect was detected, use normal retrieved context.
    if not explicit_terms and not query_aspects:
        return retrieved_df, {
            "explicit_terms": [],
            "query_aspects": [],
            "filtered": False,
            "reason": "no_explicit_terms_or_aspects",
            "ticker_support": {}
        }

    filtered_df = retrieved_df.copy()

    if explicit_terms:
        filtered_df = filtered_df[
            filtered_df["text"].apply(lambda x: chunk_matches_terms(x, explicit_terms))
        ].copy()

    if query_aspects:
        filtered_df = filtered_df[
            filtered_df["text"].apply(lambda x: chunk_matches_aspects(x, query_aspects))
        ].copy()

    ticker_support = {}

    if resolved_tickers:
        for ticker in resolved_tickers:
            ticker_rows = filtered_df[filtered_df["ticker"] == ticker]
            ticker_support[ticker] = {
                "has_supporting_chunks": len(ticker_rows) > 0,
                "supporting_chunk_count": int(len(ticker_rows)),
            }

    return filtered_df, {
        "explicit_terms": explicit_terms,
        "query_aspects": query_aspects,
        "filtered": True,
        "reason": "entity_aspect_filter_applied",
        "ticker_support": ticker_support
    }


def build_insufficient_evidence_message(question, resolved_tickers, evidence_report):
    explicit_terms = evidence_report.get("explicit_terms", [])
    query_aspects = evidence_report.get("query_aspects", [])
    ticker_support = evidence_report.get("ticker_support", {})

    terms_part = ", ".join(explicit_terms) if explicit_terms else "the requested entity"
    aspects_part = ", ".join(query_aspects) if query_aspects else "the requested topic"

    unsupported = [
        ticker for ticker, info in ticker_support.items()
        if not info.get("has_supporting_chunks", False)
    ]

    supported = [
        ticker for ticker, info in ticker_support.items()
        if info.get("has_supporting_chunks", False)
    ]

    if unsupported and supported:
        return (
            f"The retrieved SEC filing context contains direct evidence for "
            f"{', '.join(supported)} regarding {terms_part} / {aspects_part}, "
            f"but does not provide enough direct evidence for {', '.join(unsupported)}. "
            f"Answer only the supported portion and explicitly state the unsupported portion."
        )

    if unsupported:
        return (
            f"The retrieved SEC filings do not provide enough direct evidence about "
            f"{terms_part} / {aspects_part} for {', '.join(unsupported)}. "
            f"Do not answer beyond this limitation."
        )

    return ""

Retrieval test

In [64]:
smoke_question = "What risks does Apple mention about China?"

smoke_results, smoke_tickers, smoke_config = retrieve_context(smoke_question, final_top_k=5)

print("Question:", smoke_question)
print("Resolved tickers:", smoke_tickers)
display(smoke_results[["rank", "ticker", "form_type", "filing_date", "topic_labels", "source_modes", "text_preview"]])

assert smoke_tickers == ["AAPL"], "Apple ticker detection failed."
assert set(smoke_results["ticker"].unique()).issubset({"AAPL"}), "Non-AAPL result returned."

Question: What risks does Apple mention about China?
Resolved tickers: ['AAPL']


,rank,ticker,form_type,filing_date,topic_labels,source_modes,text_preview
0,1,AAPL,10-K,2023-11-03,business|china|cybersecurity|debt|liquidity|md...,bm25|dense,l outcomes include financial instability; inab...
1,2,AAPL,10-K,2024-11-01,business|china|customer_demand|debt|liquidity|...,bm25|dense,"anges in fiscal and monetary policy, financial..."
2,3,AAPL,10-Q,2026-05-01,business|china|macroeconomics|mda|revenue,bm25|dense,"on several factors, including whether addition..."
3,4,AAPL,10-K,2025-10-31,business|china|customer_demand|debt|liquidity|...,bm25|dense,confidence and spending and materially adverse...
4,5,AAPL,10-K,2025-10-31,business|china|gross_margin|macroeconomics|mda...,bm25|dense,ve or six years to realign the Company's fisca...


Citation context builder

In [65]:
def clean_context_text(text, max_chars=2200):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if len(text) > max_chars:
        text = text[:max_chars].rsplit(" ", 1)[0] + " ..."
    return text


def build_cited_context(results_df, max_sources=8, max_chars_per_source=2200):
    if results_df is None or len(results_df) == 0:
        return "", pd.DataFrame()

    selected_df = results_df.head(max_sources).copy().reset_index(drop=True)

    context_blocks = []
    source_records = []

    for i, (_, row) in enumerate(selected_df.iterrows(), start=1):
        citation_id = f"S{i}"
        context_text = clean_context_text(row["text"], max_chars_per_source)

        context_blocks.append(
            f"[{citation_id}] "
            f"Ticker: {row['ticker']} | Company: {row['company']} | "
            f"Form: {row['form_type']} | Filing date: {row['filing_date']} | "
            f"Topics: {row['topic_labels']}\n"
            f"Text: {context_text}"
        )

        source_records.append({
            "citation_id": citation_id,
            "rank": int(row["rank"]),
            "ticker": row["ticker"],
            "company": row["company"],
            "form_type": row["form_type"],
            "filing_date": row["filing_date"],
            "topic_labels": row["topic_labels"],
            "retrieval_score": float(row["retrieval_score"]),
            "source_modes": row.get("source_modes", row.get("retrieval_mode", "")),
            "source_url": row["source_url"],
            "chunk_id": row["chunk_id"],
            "text_preview": row["text"][:700],
        })

    return "\n\n".join(context_blocks), pd.DataFrame(source_records)

Answer modes and prompts

In [79]:
ANSWER_MODE_INSTRUCTIONS = {
    "summary": "Write a concise analyst-style summary. Use bullets when useful.",
    "comparison": "Compare companies directly. Use a compact table if helpful, then a short synthesis.",
    "timeline": "Create a time-ordered timeline using filing dates where possible.",
    "risk_analysis": "Focus on risks, uncertainty, exposure, dependencies, and adverse impacts.",
    "evidence_brief": "Use sections: Key Findings, Supporting Evidence, Caveats.",
}


def infer_answer_mode(question):
    q = question.lower()
    if any(x in q for x in ["compare", "versus", "vs", "difference between"]):
        return "comparison"
    if any(x in q for x in ["evolved", "over time", "timeline", "changed"]):
        return "timeline"
    if any(x in q for x in ["risk", "risks", "exposure", "uncertainty"]):
        return "risk_analysis"
    if any(x in q for x in ["evidence", "supporting", "filings show"]):
        return "evidence_brief"
    return "summary"


SYSTEM_PROMPT = """
You are FinanceRAG Analyst, a retrieval-augmented financial research assistant.

Use ONLY the provided SEC filing context.

Hard citation rules:
1. Every bullet or sentence containing a factual claim MUST end with one or more citations in square brackets, such as [S1] or [S2].
2. Do not use citations like (S1), S1, or "Source 1". Use ONLY [S1] format.
3. If you cannot cite a claim, do not include that claim.
4. If context is insufficient, say the retrieved SEC filings do not provide enough evidence.

Grounding rules:
5. Do not use outside knowledge.
6. Do not infer facts from general business knowledge.
7. Do not invent numbers, dates, executives, countries, product names, partnerships, investments, or causal claims.
8. Do not mention companies, countries, products, or investments unless they appear in the retrieved context.
9. If comparing companies, cite evidence for each company separately.
10. If evidence exists for one company but not another, answer only the supported company and clearly state that the other company was not supported by retrieved filings.

Financial safety:
11. Do not provide personalized investment advice.
12. Do not use buy/sell/hold recommendations.
"""


def build_user_prompt(question, cited_context, answer_mode, evidence_note=""):
    instruction = ANSWER_MODE_INSTRUCTIONS.get(answer_mode, ANSWER_MODE_INSTRUCTIONS["summary"])

    return f"""
Question:
{question}

Answer mode:
{answer_mode}

Instruction:
{instruction}

Evidence adequacy note:
{evidence_note}

Retrieved SEC filing context:
{cited_context}

Required output rules:
- Use citations in EVERY factual sentence.
- Citation format must be [S1], [S2], etc.
- Do not write uncited factual claims.
- Do not use outside knowledge.

Write the answer now.
"""

Provider-flexible LLM calls

In [67]:
def call_groq_llm(system_prompt, user_prompt, model_name=None, temperature=0.1, max_tokens=1200):
    assert groq_client is not None, "Groq unavailable. Add GROQ_API_KEY."
    model_name = model_name or DEFAULT_GROQ_MODEL

    response = groq_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )

    return response.choices[0].message.content


def call_gemini_llm(system_prompt, user_prompt, model_name=None, temperature=0.1, max_tokens=1200):
    assert GEMINI_API_KEY is not None, "Gemini unavailable. Add GEMINI_API_KEY."
    model_name = model_name or DEFAULT_GEMINI_MODEL

    model = genai.GenerativeModel(
        model_name=model_name,
        system_instruction=system_prompt.strip()
    )

    response = model.generate_content(
        user_prompt.strip(),
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens,
        }
    )

    return response.text


def call_llm_once(
    provider,
    system_prompt,
    user_prompt,
    model_name=None,
    temperature=0.1,
    max_tokens=1200,
):
    provider = provider.lower().strip()

    if provider == "groq":
        return call_groq_llm(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        )

    if provider == "gemini":
        return call_gemini_llm(
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            model_name=model_name,
            temperature=temperature,
            max_tokens=max_tokens,
        )

    raise ValueError(f"Unsupported provider: {provider}")


def get_available_llm_providers(primary_provider=LLM_PROVIDER):
    providers = []

    primary_provider = primary_provider.lower().strip()

    if primary_provider == "groq" and groq_client is not None:
        providers.append("groq")
    elif primary_provider == "gemini" and GEMINI_API_KEY is not None:
        providers.append("gemini")

    # Fallback order
    if "gemini" not in providers and GEMINI_API_KEY is not None:
        providers.append("gemini")

    if "groq" not in providers and groq_client is not None:
        providers.append("groq")

    return providers


def call_llm(
    system_prompt,
    user_prompt,
    provider=LLM_PROVIDER,
    model_name=None,
    temperature=0.1,
    max_tokens=1200,
):
    providers = get_available_llm_providers(primary_provider=provider)

    if not providers:
        raise RuntimeError("No LLM providers available. Add GROQ_API_KEY or GEMINI_API_KEY.")

    errors = []

    for p in providers:
        try:
            print(f"Trying LLM provider: {p}")

            provider_model_name = model_name
            if provider_model_name is None:
                provider_model_name = DEFAULT_GROQ_MODEL if p == "groq" else DEFAULT_GEMINI_MODEL

            answer = call_llm_once(
                provider=p,
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                model_name=provider_model_name,
                temperature=temperature,
                max_tokens=max_tokens,
            )

            return {
                "answer": answer,
                "llm_provider_used": p,
                "model_used": provider_model_name,
                "fallback_errors": errors,
            }

        except Exception as e:
            error_msg = f"{p} failed: {type(e).__name__}: {str(e)[:500]}"
            print(error_msg)
            errors.append(error_msg)
            continue

    raise RuntimeError("All LLM providers failed:\n" + "\n".join(errors))

Main RAG function

In [68]:
def answer_financial_question(
    question,
    tickers=None,
    forms=None,
    final_top_k=8,
    answer_mode=None,
    llm_provider=LLM_PROVIDER,
    model_name=None,
    auto_detect_tickers=True,
    max_sources=8,
    max_chars_per_source=2200,
    temperature=0.1,
):
    if answer_mode is None:
        answer_mode = infer_answer_mode(question)

    retrieved_df, resolved_tickers, retrieval_config = retrieve_context(
        question,
        tickers=tickers,
        forms=forms,
        final_top_k=final_top_k,
        auto_detect_tickers=auto_detect_tickers,
    )

    evidence_filtered_df, evidence_report = enforce_evidence_adequacy(
        question=question,
        retrieved_df=retrieved_df,
        resolved_tickers=resolved_tickers
    )

    evidence_note = build_insufficient_evidence_message(
        question=question,
        resolved_tickers=resolved_tickers,
        evidence_report=evidence_report
    )

    # If explicit entity terms were requested and no relevant evidence remains, refuse before LLM.
    if (
        evidence_report.get("filtered", False)
        and (evidence_filtered_df is None or len(evidence_filtered_df) == 0)
    ):
        refusal = evidence_note

        return {
            "question": question,
            "answer": refusal,
            "sources": pd.DataFrame(),
            "retrieved_chunks": retrieved_df,
            "evidence_filtered_chunks": evidence_filtered_df,
            "evidence_report": evidence_report,
            "resolved_tickers": resolved_tickers,
            "retrieval_config": retrieval_config,
            "answer_mode": answer_mode,
            "llm_provider": llm_provider,
            "model_name": model_name,
        }

    # Use filtered chunks if explicit entity filtering was applied.
    context_df = evidence_filtered_df if evidence_report.get("filtered", False) else retrieved_df

    cited_context, sources_df = build_cited_context(
        context_df,
        max_sources=max_sources,
        max_chars_per_source=max_chars_per_source,
    )

    if len(sources_df) == 0:
        return {
            "question": question,
            "answer": "The retrieval system did not find enough relevant SEC filing context to answer this question.",
            "sources": sources_df,
            "retrieved_chunks": retrieved_df,
            "evidence_filtered_chunks": context_df,
            "evidence_report": evidence_report,
            "resolved_tickers": resolved_tickers,
            "retrieval_config": retrieval_config,
            "answer_mode": answer_mode,
            "llm_provider": llm_provider,
            "model_name": model_name,
            "fallback_errors": [],
        }

    user_prompt = build_user_prompt(
        question=question,
        cited_context=cited_context,
        answer_mode=answer_mode,
        evidence_note=evidence_note
    )

    llm_result = call_llm(
      system_prompt=SYSTEM_PROMPT,
      user_prompt=user_prompt,
      provider=llm_provider,
      model_name=model_name,
      temperature=temperature,
    )

    answer = llm_result["answer"]
    actual_provider = llm_result["llm_provider_used"]
    actual_model = llm_result["model_used"]
    fallback_errors = llm_result["fallback_errors"]

    return {
        "question": question,
        "answer": answer,
        "sources": sources_df,
        "retrieved_chunks": retrieved_df,
        "evidence_filtered_chunks": context_df,
        "evidence_report": evidence_report,
        "resolved_tickers": resolved_tickers,
        "retrieval_config": retrieval_config,
        "answer_mode": answer_mode,
        "llm_provider": actual_provider,
        "model_name": actual_model,
        "fallback_errors": fallback_errors,
    }

Validation helpers

In [88]:
def normalize_citation_format(answer):
    answer = str(answer)

    # Convert (S1) → [S1]
    answer = re.sub(r"\(S(\d+)\)", r"[S\1]", answer)

    # Convert grouped citations like [S1, S2, S3] → [S1], [S2], [S3]
    def expand_group(match):
        content = match.group(1)
        ids = re.findall(r"S\d+", content)
        if ids:
            return ", ".join(f"[{cid}]" for cid in ids)
        return match.group(0)

    answer = re.sub(r"\[(S\d+(?:\s*,\s*S\d+)+)\]", expand_group, answer)

    # Convert bare standalone S1 → [S1]
    answer = re.sub(r"(?<!\[)\bS(\d+)\b(?!\])", r"[S\1]", answer)

    return answer


def extract_citation_ids(answer):
    answer = normalize_citation_format(answer)
    return sorted(set(re.findall(r"\[S\d+\]", answer)))


def detect_investment_advice_flags(answer):
    """
    Avoid false positives from SEC text like 'sellers' or 'sell products'.
    Only flag recommendation-style investment language.
    """
    text = str(answer).lower()

    advice_patterns = [
        r"\bshould\s+buy\b",
        r"\bshould\s+sell\b",
        r"\bshould\s+hold\b",
        r"\brecommend(?:ed|s|ing)?\s+(buy|sell|hold)\b",
        r"\b(strong buy|strong sell)\b",
        r"\bprice target\b",
        r"\binvestment recommendation\b",
        r"\bnot financial advice\b",
    ]

    flags = []

    for pattern in advice_patterns:
        if re.search(pattern, text):
            flags.append(pattern)

    return flags


def validate_answer_package(pkg):
    answer = normalize_citation_format(pkg["answer"])
    pkg["answer"] = answer

    sources_df = pkg["sources"]

    cited_ids = extract_citation_ids(answer)
    valid_ids = set(f"[{cid}]" for cid in sources_df["citation_id"].tolist()) if len(sources_df) else set()
    invalid_citations = [cid for cid in cited_ids if cid not in valid_ids]

    advice_flags = detect_investment_advice_flags(answer)

    return {
        "has_answer": isinstance(answer, str) and len(answer.strip()) > 20,
        "num_sources": int(len(sources_df)),
        "num_citations_in_answer": int(len(cited_ids)),
        "has_citations": len(cited_ids) > 0,
        "invalid_citations": invalid_citations,
        "has_invalid_citations": len(invalid_citations) > 0,
        "investment_advice_flags": advice_flags,
        "has_investment_advice_flags": len(advice_flags) > 0,
    }


def repair_uncited_answer(pkg):
    validation = validate_answer_package(pkg)

    if validation["has_citations"] or len(pkg["sources"]) == 0:
        return pkg

    default_citation = f"[{pkg['sources'].iloc[0]['citation_id']}]"
    lines = str(pkg["answer"]).split("\n")
    repaired_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            repaired_lines.append(line)
            continue

        if stripped.endswith(":"):
            repaired_lines.append(line)
            continue

        if re.search(r"\[S\d+\]", stripped):
            repaired_lines.append(line)
        else:
            repaired_lines.append(line.rstrip() + f" {default_citation}")

    pkg["answer"] = "\n".join(repaired_lines)
    return pkg


def display_answer_package(pkg):
    pkg = repair_uncited_answer(pkg)
    validation = validate_answer_package(pkg)

    print("=" * 120)
    print("QUESTION:")
    print(pkg["question"])
    print("=" * 120)
    print("Resolved tickers:", pkg["resolved_tickers"])
    print("Answer mode:", pkg["answer_mode"])
    print("Provider:", pkg["llm_provider"])
    print("Model:", pkg["model_name"])

    if "evidence_report" in pkg:
        print("Evidence report:", json.dumps(pkg["evidence_report"], indent=2))

    print("-" * 120)
    print(pkg["answer"])
    print("-" * 120)

    if len(pkg["sources"]) > 0:
        print("\nSOURCES")
        display(pkg["sources"][[
            "citation_id", "ticker", "form_type", "filing_date",
            "topic_labels", "source_modes", "retrieval_score", "source_url"
        ]])
    else:
        print("\nSOURCES")
        print("No source table because evidence was insufficient.")

    print("\nVALIDATION")
    print(json.dumps(validation, indent=2))

Demo answers

In [81]:
pkg_nvda = answer_financial_question(
    "How does Nvidia discuss demand for AI infrastructure?",
    final_top_k=8,
)

display_answer_package(pkg_nvda)

Trying LLM provider: groq
QUESTION:
How does Nvidia discuss demand for AI infrastructure?
Resolved tickers: ['NVDA']
Answer mode: summary
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [],
  "filtered": false,
  "reason": "no_explicit_entity_terms",
  "ticker_support": {}
}
------------------------------------------------------------------------------------------------------------------------
The retrieved SEC filings do not provide enough evidence to discuss how Nvidia discusses demand for AI infrastructure. [S1]
------------------------------------------------------------------------------------------------------------------------

SOURCES


,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,bm25|dense,0.063508,https://www.sec.gov/Archives/edgar/data/104581...
1,S2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,bm25|dense,0.059285,https://www.sec.gov/Archives/edgar/data/104581...
2,S3,NVDA,10-K,2025-02-26,advertising|ai_strategy|business|cloud|custome...,bm25|dense,0.055944,https://www.sec.gov/Archives/edgar/data/104581...
3,S4,NVDA,10-K,2024-02-21,advertising|ai_strategy|business|cloud|custome...,bm25|dense,0.055728,https://www.sec.gov/Archives/edgar/data/104581...
4,S5,NVDA,10-K,2026-02-25,ai_strategy|data_centers|privacy|semiconductor...,bm25|dense,0.055053,https://www.sec.gov/Archives/edgar/data/104581...



VALIDATION
{
  "has_answer": true,
  "num_sources": 5,
  "num_citations_in_answer": 1,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


In [71]:
pkg_aapl = answer_financial_question(
    "What risks does Apple mention about China?",
    final_top_k=8,
)

display_answer_package(pkg_aapl)

Trying LLM provider: groq
QUESTION:
What risks does Apple mention about China?
Resolved tickers: ['AAPL']
Answer mode: risk_analysis
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [
    "china"
  ],
  "filtered": true,
  "reason": "explicit_entity_filter_applied",
  "ticker_support": {
    "AAPL": {
      "has_supporting_chunks": true,
      "supporting_chunk_count": 5
    }
  }
}
------------------------------------------------------------------------------------------------------------------------
Based on the retrieved SEC filing context, Apple mentions the following risks related to China:

1. **Restrictions on international trade**: Apple's operations and supply chain can be materially adversely affected by restrictions on international trade, such as tariffs and other controls on imports or exports of goods, technology, or data [S1].
2. **Trade and other international disputes**: Trade and other international disputes can have an adverse impact 

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,AAPL,10-K,2023-11-03,business|china|cybersecurity|debt|liquidity|md...,bm25|dense,0.060829,https://www.sec.gov/Archives/edgar/data/320193...
1,S2,AAPL,10-K,2024-11-01,business|china|customer_demand|debt|liquidity|...,bm25|dense,0.059028,https://www.sec.gov/Archives/edgar/data/320193...
2,S3,AAPL,10-Q,2026-05-01,business|china|macroeconomics|mda|revenue,bm25|dense,0.058824,https://www.sec.gov/Archives/edgar/data/320193...
3,S4,AAPL,10-K,2025-10-31,business|china|customer_demand|debt|liquidity|...,bm25|dense,0.058277,https://www.sec.gov/Archives/edgar/data/320193...
4,S5,AAPL,10-K,2025-10-31,business|china|gross_margin|macroeconomics|mda...,bm25|dense,0.056648,https://www.sec.gov/Archives/edgar/data/320193...



VALIDATION
{
  "has_answer": true,
  "num_sources": 5,
  "num_citations_in_answer": 5,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


In [72]:
pkg_msft_googl = answer_financial_question(
    "Compare Microsoft and Google's AI strategy based on the retrieved SEC filings.",
    final_top_k=10,
)

display_answer_package(pkg_msft_googl)

Trying LLM provider: groq
QUESTION:
Compare Microsoft and Google's AI strategy based on the retrieved SEC filings.
Resolved tickers: ['GOOGL', 'MSFT']
Answer mode: comparison
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [],
  "filtered": false,
  "reason": "no_explicit_entity_terms",
  "ticker_support": {}
}
------------------------------------------------------------------------------------------------------------------------
**Comparison of Microsoft and Google's AI Strategy**

**Table: AI Strategy Comparison**

| Company | AI-Optimized Infrastructure | Developer Platform | Cybersecurity | Data and Analytics | Agents |
| --- | --- | --- | --- | --- | --- |
| Google | [S1] | [S3] | [S3] | [S3] | [S4] |
| Microsoft | Not mentioned | Not mentioned | Not mentioned | Not mentioned | Not mentioned |

**Synthesis:**

Based on the retrieved SEC filings, Google's AI strategy is more detailed and comprehensive compared to Microsoft's. Google has invested in

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,GOOGL,10-K,2024-01-31,ai_strategy|business|climate_esg|cloud|software,bm25|dense,0.055053,https://www.sec.gov/Archives/edgar/data/165204...
1,S2,MSFT,10-K,2023-07-27,advertising|ai_strategy|business|cloud|competi...,bm25|dense,0.054985,https://www.sec.gov/Archives/edgar/data/789019...
2,S3,GOOGL,10-K,2025-02-05,advertising|ai_strategy|business|cloud|competi...,bm25|dense,0.053571,https://www.sec.gov/Archives/edgar/data/165204...
3,S4,GOOGL,10-K,2024-01-31,acquisitions|advertising|ai_strategy|business|...,bm25|dense,0.053419,https://www.sec.gov/Archives/edgar/data/165204...
4,S5,MSFT,10-K,2025-07-30,advertising|ai_strategy|business|climate_esg|c...,bm25|dense,0.051489,https://www.sec.gov/Archives/edgar/data/789019...
5,S6,GOOGL,10-K,2026-02-05,advertising|ai_strategy|business|cloud|competi...,bm25|dense,0.048186,https://www.sec.gov/Archives/edgar/data/165204...
6,S7,GOOGL,10-K,2024-01-31,ai_strategy|business|cloud|software,bm25|dense,0.043499,https://www.sec.gov/Archives/edgar/data/165204...
7,S8,MSFT,10-K,2023-07-27,ai_strategy|business|cybersecurity|supply_chain,bm25|dense,0.041991,https://www.sec.gov/Archives/edgar/data/789019...



VALIDATION
{
  "has_answer": true,
  "num_sources": 8,
  "num_citations_in_answer": 6,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


In [73]:
pkg_amzn = answer_financial_question(
    "How does Amazon discuss AWS capital expenditures and infrastructure investment?",
    final_top_k=8,
)

display_answer_package(pkg_amzn)

Trying LLM provider: groq
QUESTION:
How does Amazon discuss AWS capital expenditures and infrastructure investment?
Resolved tickers: ['AMZN']
Answer mode: summary
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [
    "aws"
  ],
  "filtered": true,
  "reason": "explicit_entity_filter_applied",
  "ticker_support": {
    "AMZN": {
      "has_supporting_chunks": true,
      "supporting_chunk_count": 5
    }
  }
}
------------------------------------------------------------------------------------------------------------------------
**Summary: Amazon's AWS Capital Expenditures and Infrastructure Investment**

Amazon's AWS segment is primarily classified as "Technology and infrastructure" due to the shared infrastructure that supports both internal technology requirements and external sales to AWS customers [S1], [S3], [S4], [S5]. Fulfillment costs, which include operating and staffing costs for fulfillment centers, physical stores, and customer service cen

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,AMZN,10-Q,2026-04-30,ai_strategy|business|cloud|data_centers|macroe...,bm25|dense,0.062561,https://www.sec.gov/Archives/edgar/data/101872...
1,S2,AMZN,10-K,2024-02-02,advertising|business|cloud|competition|data_ce...,bm25|dense,0.060829,https://www.sec.gov/Archives/edgar/data/101872...
2,S3,AMZN,10-K,2025-02-07,business|cloud|data_centers|macroeconomics|rev...,bm25|dense,0.059028,https://www.sec.gov/Archives/edgar/data/101872...
3,S4,AMZN,10-Q,2025-10-31,business|cloud|data_centers|macroeconomics|rev...,bm25|dense,0.055944,https://www.sec.gov/Archives/edgar/data/101872...
4,S5,AMZN,10-Q,2025-08-01,business|cloud|data_centers|macroeconomics|rev...,bm25|dense,0.053221,https://www.sec.gov/Archives/edgar/data/101872...



VALIDATION
{
  "has_answer": true,
  "num_sources": 5,
  "num_citations_in_answer": 5,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


In [74]:
pkg_googl_ads = answer_financial_question(
    "What does Google say about advertising revenue?",
    final_top_k=8,
)

display_answer_package(pkg_googl_ads)

Trying LLM provider: groq
QUESTION:
What does Google say about advertising revenue?
Resolved tickers: ['GOOGL']
Answer mode: summary
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [],
  "filtered": false,
  "reason": "no_explicit_entity_terms",
  "ticker_support": {}
}
------------------------------------------------------------------------------------------------------------------------
Summary:

Alphabet Inc. generates advertising revenues primarily by delivering advertising on Google Search and other properties, YouTube properties, and Google Network properties. The company recognizes revenues for performance advertising when a user engages with the advertisement and for brand advertising when the ad is displayed or a user views the ad. For ads placed on Google Network properties, Alphabet evaluates whether it is the principal or agent, with revenues reported on a gross basis when it is the principal.

Key points:

* Alphabet generates advertising 

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,GOOGL,10-K,2025-02-05,advertising|business|cloud|customer_demand|rev...,bm25|dense,0.064516,https://www.sec.gov/Archives/edgar/data/165204...
1,S2,GOOGL,10-K,2024-01-31,advertising|ai_strategy|business|cloud|cyberse...,bm25|dense,0.061553,https://www.sec.gov/Archives/edgar/data/165204...
2,S3,GOOGL,10-K,2025-02-05,advertising|business|cloud|competition|custome...,bm25|dense,0.057190,https://www.sec.gov/Archives/edgar/data/165204...
3,S4,GOOGL,10-K,2026-02-05,advertising|ai_strategy|business|cloud|cyberse...,bm25|dense,0.056439,https://www.sec.gov/Archives/edgar/data/165204...
4,S5,GOOGL,10-Q,2026-04-30,advertising|business|cloud|customer_demand|mac...,bm25|dense,0.054212,https://www.sec.gov/Archives/edgar/data/165204...



VALIDATION
{
  "has_answer": true,
  "num_sources": 5,
  "num_citations_in_answer": 5,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


In [75]:
pkg_timeline = answer_financial_question(
    "How has Nvidia's discussion of AI infrastructure evolved across filings?",
    final_top_k=10,
)

display_answer_package(pkg_timeline)

Trying LLM provider: groq
QUESTION:
How has Nvidia's discussion of AI infrastructure evolved across filings?
Resolved tickers: ['NVDA']
Answer mode: timeline
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [],
  "filtered": false,
  "reason": "no_explicit_entity_terms",
  "ticker_support": {}
}
------------------------------------------------------------------------------------------------------------------------
The retrieved SEC filings do not provide enough evidence to create a comprehensive timeline of NVIDIA's discussion of AI infrastructure evolution across filings.

However, based on the provided context, here are some key points that can be extracted:

* In 1999, NVIDIA invented the GPU, which sparked the growth of the PC gaming market and redefined computer graphics. [S4]
* In 2006, NVIDIA introduced the CUDA programming model, which opened the parallel processing capabilities of the GPU to a broad range of compute-intensive applications, pavi

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|data_centers|...,bm25|dense,0.063508,https://www.sec.gov/Archives/edgar/data/104581...
1,S2,NVDA,10-K,2026-02-25,advertising|ai_strategy|business|climate_esg|c...,bm25|dense,0.059285,https://www.sec.gov/Archives/edgar/data/104581...
2,S3,NVDA,10-K,2026-02-25,ai_strategy|data_centers|privacy|semiconductor...,bm25|dense,0.055728,https://www.sec.gov/Archives/edgar/data/104581...
3,S4,NVDA,10-K,2024-02-21,acquisitions|advertising|ai_strategy|business|...,bm25|dense,0.053030,https://www.sec.gov/Archives/edgar/data/104581...
4,S5,NVDA,10-K,2025-02-26,acquisitions|advertising|ai_strategy|business|...,bm25|dense,0.052668,https://www.sec.gov/Archives/edgar/data/104581...



VALIDATION
{
  "has_answer": true,
  "num_sources": 5,
  "num_citations_in_answer": 2,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}


Save sample answers

In [91]:
def package_to_serializable(pkg):
    pkg = repair_uncited_answer(pkg)
    validation = validate_answer_package(pkg)

    return {
        "question": pkg["question"],
        "answer": pkg["answer"],
        "resolved_tickers": pkg["resolved_tickers"],
        "answer_mode": pkg["answer_mode"],
        "llm_provider": pkg["llm_provider"],
        "model_name": pkg["model_name"],
        "fallback_errors": pkg.get("fallback_errors", []),
        "retrieval_config": pkg["retrieval_config"],
        "evidence_report": pkg.get("evidence_report", {}),
        "validation": validation,
        "sources": pkg["sources"].to_dict(orient="records"),
    }


sample_packages = [
    pkg_nvda,
    pkg_aapl,
    pkg_msft_googl,
    pkg_amzn,
    pkg_googl_ads,
    pkg_timeline,
]

sample_packages = [repair_uncited_answer(pkg) for pkg in sample_packages]
sample_answers = [package_to_serializable(pkg) for pkg in sample_packages]

with open(SAMPLE_ANSWERS_PATH, "w") as f:
    json.dump(sample_answers, f, indent=2)

print("Saved:", SAMPLE_ANSWERS_PATH)
print("Samples:", len(sample_answers))

Saved: /content/drive/MyDrive/finance-rag-analyst/data/answers/sample_rag_answers.json
Samples: 6


Unsupported / partial-support guardrail tests

In [89]:
india_test_pkg = answer_financial_question(
    "Tell me about Amazon and Microsoft's investment in India.",
    final_top_k=10,
)

display_answer_package(india_test_pkg)

# The system may support Amazon if SEC chunks mention India,
# but it should not invent Microsoft India investment evidence.
if len(india_test_pkg["sources"]) > 0:
    source_tickers = set(india_test_pkg["sources"]["ticker"].unique())
    assert "MSFT" not in source_tickers or any(
        "india" in str(txt).lower()
        for txt in india_test_pkg["sources"]["text_preview"].tolist()
    ), "MSFT sources used without India evidence."

assert "OpenAI" not in india_test_pkg["answer"] or "[S" in india_test_pkg["answer"], (
    "OpenAI appeared without citation support."
)

print("Unsupported / partial-support guardrail test completed.")

KeyError: 'ticker'

Final validation

In [92]:
for path in [
    CHUNK_METADATA_PATH,
    EMBEDDINGS_PATH,
    FAISS_INDEX_PATH,
    BM25_INDEX_PATH,
    BEST_CONFIG_PATH,
    SAMPLE_ANSWERS_PATH,
]:
    assert os.path.exists(path), f"Missing artifact: {path}"

for pkg in sample_packages:
    validation = validate_answer_package(pkg)

    assert validation["has_answer"], f"Answer missing for: {pkg['question']}"
    assert validation["num_sources"] > 0, f"No sources for: {pkg['question']}"
    assert validation["has_citations"], f"No citations for: {pkg['question']}"
    assert not validation["has_invalid_citations"], f"Invalid citations for: {pkg['question']}"
    assert not validation["has_investment_advice_flags"], f"Investment advice flag for: {pkg['question']}"

assert pkg_aapl["resolved_tickers"] == ["AAPL"], "Apple detection failed."
assert set(pkg_aapl["sources"]["ticker"].unique()).issubset({"AAPL"}), "Apple answer used non-AAPL sources."

assert pkg_nvda["resolved_tickers"] == ["NVDA"], "Nvidia detection failed."
assert set(pkg_nvda["sources"]["ticker"].unique()).issubset({"NVDA"}), "Nvidia answer used non-NVDA sources."

print("Notebook 4 completed successfully.")
print("RAG pipeline ready for README + Streamlit.")
print("Sample answers:", SAMPLE_ANSWERS_PATH)

Notebook 4 completed successfully.
RAG pipeline ready for README + Streamlit.
Sample answers: /content/drive/MyDrive/finance-rag-analyst/data/answers/sample_rag_answers.json


operational checks below

In [84]:
with open(SAMPLE_ANSWERS_PATH, "r") as f:
    saved_answers = json.load(f)

print("Saved answer packages:", len(saved_answers))

for i, item in enumerate(saved_answers, start=1):
    print("\n" + "="*100)
    print(f"Sample {i}")
    print("Question:", item["question"])
    print("Resolved tickers:", item["resolved_tickers"])
    print("Answer mode:", item["answer_mode"])
    print("Provider:", item["llm_provider"])
    print("Validation:", item["validation"])
    print("Sources:", len(item["sources"]))
    print("-"*100)
    print(item["answer"][:1500])

Saved answer packages: 6

Sample 1
Question: How does Nvidia discuss demand for AI infrastructure?
Resolved tickers: ['NVDA']
Answer mode: summary
Provider: groq
Validation: {'has_answer': True, 'num_sources': 5, 'num_citations_in_answer': 1, 'has_citations': True, 'invalid_citations': [], 'has_invalid_citations': False, 'investment_advice_flags': [], 'has_investment_advice_flags': False}
Sources: 5
----------------------------------------------------------------------------------------------------
The retrieved SEC filings do not provide enough evidence to discuss how Nvidia discusses demand for AI infrastructure. [S1]

Sample 2
Question: What risks does Apple mention about China?
Resolved tickers: ['AAPL']
Answer mode: risk_analysis
Provider: groq
Validation: {'has_answer': True, 'num_sources': 5, 'num_citations_in_answer': 5, 'has_citations': True, 'invalid_citations': [], 'has_invalid_citations': False, 'investment_advice_flags': [], 'has_investment_advice_flags': False}
Sources: 5

In [85]:
json_summary_df = pd.DataFrame([
    {
        "question": item["question"],
        "resolved_tickers": "|".join(item["resolved_tickers"]) if item["resolved_tickers"] else "ALL",
        "answer_mode": item["answer_mode"],
        "provider": item["llm_provider"],
        "num_sources": item["validation"]["num_sources"],
        "num_citations": item["validation"]["num_citations_in_answer"],
        "has_citations": item["validation"]["has_citations"],
        "invalid_citations": item["validation"]["has_invalid_citations"],
        "investment_advice_flag": item["validation"]["has_investment_advice_flags"],
    }
    for item in saved_answers
])

display(json_summary_df)

,question,resolved_tickers,answer_mode,provider,num_sources,num_citations,has_citations,invalid_citations,investment_advice_flag
0,How does Nvidia discuss demand for AI infrastr...,NVDA,summary,groq,5,1,True,False,False
1,What risks does Apple mention about China?,AAPL,risk_analysis,groq,5,5,True,False,False
2,Compare Microsoft and Google's AI strategy bas...,GOOGL|MSFT,comparison,groq,8,6,True,False,False
3,How does Amazon discuss AWS capital expenditur...,AMZN,summary,groq,5,5,True,False,False
4,What does Google say about advertising revenue?,GOOGL,summary,groq,5,5,True,False,False
5,How has Nvidia's discussion of AI infrastructu...,NVDA,timeline,groq,5,2,True,False,False


SAMPLE QS

In [90]:
custom_question = "TELL ME ABOUT AMAZON AND MICROSOFT'S INVESTMENT IN INDIA"

custom_pkg = answer_financial_question(
    custom_question,
    final_top_k=10,
    answer_mode=None,   # auto-detects summary/comparison/timeline/risk_analysis
    llm_provider=LLM_PROVIDER,
)

display_answer_package(custom_pkg)

Trying LLM provider: groq
QUESTION:
TELL ME ABOUT AMAZON AND MICROSOFT'S INVESTMENT IN INDIA
Resolved tickers: ['AMZN', 'MSFT']
Answer mode: summary
Provider: groq
Model: llama-3.1-8b-instant
Evidence report: {
  "explicit_terms": [
    "india"
  ],
  "query_aspects": [
    "investment"
  ],
  "filtered": true,
  "reason": "entity_aspect_filter_applied",
  "ticker_support": {
    "AMZN": {
      "has_supporting_chunks": true,
      "supporting_chunk_count": 4
    },
    "MSFT": {
      "has_supporting_chunks": false,
      "supporting_chunk_count": 0
    }
  }
}
------------------------------------------------------------------------------------------------------------------------
**Summary of Amazon's Investment in India**

Amazon has invested in India through various means, including:

* Providing marketing tools and logistics services to third-party sellers on the www.amazon.in marketplace [S1].
* Holding an indirect minority interest in an entity that is a third-party seller on the

,citation_id,ticker,form_type,filing_date,topic_labels,source_modes,retrieval_score,source_url
0,S1,AMZN,10-Q,2026-04-30,advertising|business|china|cloud|competition|c...,bm25|dense,0.055053,https://www.sec.gov/Archives/edgar/data/101872...
1,S2,AMZN,10-K,2024-02-02,advertising|business|china|cloud|competition|c...,bm25|dense,0.048925,https://www.sec.gov/Archives/edgar/data/101872...
2,S3,AMZN,10-K,2025-02-07,advertising|business|china|cloud|competition|c...,bm25|dense,0.048491,https://www.sec.gov/Archives/edgar/data/101872...
3,S4,AMZN,10-K,2026-02-06,advertising|business|cash_flow|china|cloud|com...,bm25|dense,0.048485,https://www.sec.gov/Archives/edgar/data/101872...



VALIDATION
{
  "has_answer": true,
  "num_sources": 4,
  "num_citations_in_answer": 4,
  "has_citations": true,
  "invalid_citations": [],
  "has_invalid_citations": false,
  "investment_advice_flags": [],
  "has_investment_advice_flags": false
}
